# Standalone Stock Prediction Model Training (LSTM & XGBoost)
This notebook contains everything you need. You do not need to upload any other files!

Just run all the cells below. The first few cells will automatically create the necessary Python scripts in the Colab environment, and the final cell will run the training.

In [ ]:
%%writefile requirements.txt
# ── Core ML ───────────────────────────────────────────────────────────────────
numpy==2.1.3
pandas==2.2.3
scikit-learn==1.5.2
tensorflow
keras

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgboost==2.1.3

# ── Data ──────────────────────────────────────────────────────────────────────
yfinance==1.5.2
pyarrow==18.1.0

# ── Experiment Tracking ───────────────────────────────────────────────────────
mlflow==2.19.0

# ── API ───────────────────────────────────────────────────────────────────────
fastapi==0.115.6
uvicorn[standard]==0.32.1
python-multipart==0.0.19

# ── Plotting (for training evaluation plots only) ─────────────────────────────
matplotlib==3.9.4

# ── Testing ───────────────────────────────────────────────────────────────────
pytest==8.3.4
pytest-cov==6.0.0


In [ ]:
%%writefile utils.py
"""
utils.py — Core ML helpers: data fetch, sequences, metrics, MC-Dropout forecast.

Fixes applied vs v1:
  • Scaler is NEVER fit_transform'd on full data — callers must pass pre-split arrays.
  • MASE replaces epsilon-hacked MAPE.
  • MC-Dropout forecast returns mean + confidence band instead of a single point.
  • Moving averages are computed on a given series slice, not globally.
"""

from __future__ import annotations

import os
import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path
from datetime import date
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# ── Data Fetching ──────────────────────────────────────────────────────────────

def fetch_stock_data(ticker: str, period: str = "5y",
                     cache_dir: str = "data") -> pd.DataFrame:
    """
    Download historical OHLCV data via yfinance.
    Saves a dated Parquet snapshot for reproducibility.
    """
    Path(cache_dir).mkdir(exist_ok=True)
    snap_path = Path(cache_dir) / f"{ticker}_{period}_{date.today()}.parquet"

    if snap_path.exists():
        df = pd.read_parquet(snap_path)
        print(f"  [cache] Loaded {ticker} ({period}) from {snap_path}")
        return df

    df = yf.download(ticker, period=period, auto_adjust=True, progress=False)
    # Flatten MultiIndex if present (yfinance ≥ 0.2 may return one)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.dropna(inplace=True)
    df.to_parquet(snap_path)
    print(f"  [cache] Saved {ticker} snapshot → {snap_path}")
    return df


# ── Feature Engineering ────────────────────────────────────────────────────────

def add_moving_averages(series: pd.Series) -> pd.DataFrame:
    """
    Compute MA20 / MA50 / MA200 causally on a 1-D price series.
    Returns a DataFrame aligned to the series index.
    NOTE: Do NOT call this on the full dataframe before split — compute it
          per-split slice to guarantee no future information leakage if MAs
          are ever used as LSTM features.
    """
    df = pd.DataFrame({"Close": series})
    df["MA20"]  = series.rolling(window=20,  min_periods=1).mean()
    df["MA50"]  = series.rolling(window=50,  min_periods=1).mean()
    df["MA200"] = series.rolling(window=200, min_periods=1).mean()
    return df


def build_technical_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute technical indicators for multivariate LSTM input.
    Operates on a dataframe containing at least 'Close' and 'Volume'.
    """
    out = df.copy()
    
    # RSI
    delta = out["Close"].diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.rolling(14, min_periods=1).mean()
    avg_loss = loss.rolling(14, min_periods=1).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    out["RSI"] = 100 - (100 / (1 + rs))
    
    # MACD
    ema12 = out["Close"].ewm(span=12, adjust=False).mean()
    ema26 = out["Close"].ewm(span=26, adjust=False).mean()
    out["MACD"] = ema12 - ema26
    
    # Returns
    out["Return"] = out["Close"].pct_change()
    
    out.dropna(inplace=True)
    return out


def build_lag_features(series: pd.Series, n_lags: int = 60) -> pd.DataFrame:
    """
    Build a tabular lag feature matrix for tree-based models (XGBoost).
    Also includes: RSI-14, MACD, daily return.
    All features are computed causally (no look-ahead).
    """
    df = pd.DataFrame({"close": series.values}, index=series.index)

    # Lag features
    for lag in range(1, n_lags + 1):
        df[f"lag_{lag}"] = df["close"].shift(lag)

    # Technical indicators
    delta    = df["close"].diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.rolling(14, min_periods=1).mean()
    avg_loss = loss.rolling(14, min_periods=1).mean()
    rs       = avg_gain / (avg_loss + 1e-9)
    df["rsi"]     = 100 - (100 / (1 + rs))
    ema12         = df["close"].ewm(span=12, adjust=False).mean()
    ema26         = df["close"].ewm(span=26, adjust=False).mean()
    df["macd"]    = ema12 - ema26
    df["return1"] = df["close"].pct_change()

    df.dropna(inplace=True)
    return df


# ── Sequence Preparation for LSTM ─────────────────────────────────────────────

def create_sequences(features_scaled: np.ndarray, target_scaled: np.ndarray, lookback: int = 60):
    """
    Create (X, y) sliding-window sequences from SCALED arrays.

    Args:
        features_scaled: shape (N, num_features) — already scaled.
        target_scaled:   shape (N, 1) — already scaled.
        lookback:        window length.

    Returns:
        X: shape (N-lookback, lookback, num_features)
        y: shape (N-lookback,)
    """
    assert features_scaled.ndim == 2, "features_scaled must be shape (N, num_features)"
    assert target_scaled.ndim == 2 and target_scaled.shape[1] == 1, "target_scaled must be shape (N, 1)"
    assert len(features_scaled) == len(target_scaled), "features and target must have same length"
    
    X, y = [], []
    for i in range(lookback, len(features_scaled)):
        X.append(features_scaled[i - lookback: i, :])
        y.append(target_scaled[i, 0])
        
    X = np.array(X)
    y = np.array(y)
    return X, y


# ── Metrics ────────────────────────────────────────────────────────────────────

def returns_to_prices(last_price: float, returns: np.ndarray) -> np.ndarray:
    """
    Convert an array of returns back into absolute prices given the last known price.
    Prices[0] = last_price * (1 + returns[0])
    Prices[1] = Prices[0] * (1 + returns[1])
    """
    prices = [last_price * (1 + returns[0])]
    for r in returns[1:]:
        prices.append(prices[-1] * (1 + r))
    return np.array(prices)


def compute_metrics(actual: np.ndarray, predicted: np.ndarray,
                    naive: np.ndarray | None = None) -> dict:
    """
    Compute evaluation metrics.

    Args:
        actual:    true prices (1-D)
        predicted: model predictions (1-D)
        naive:     naive (lag-1) predictions for MASE denominator.
                   If None, MASE is skipped.

    Returns dict with: RMSE, MAE, R², MASE (if naive provided).
    """
    actual    = actual.flatten()
    predicted = predicted.flatten()

    rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    mae  = float(mean_absolute_error(actual, predicted))
    r2   = float(r2_score(actual, predicted))

    out = {"RMSE": rmse, "MAE": mae, "R²": r2}

    if naive is not None:
        naive = naive.flatten()
        # MASE = MAE(model) / MAE(naive)
        mae_naive = float(mean_absolute_error(actual, naive))
        out["MASE"] = mae / (mae_naive + 1e-9)

    return out


def naive_forecast(prices: np.ndarray) -> np.ndarray:
    """Shift-by-1 naive baseline: predict tomorrow = today."""
    return prices[:-1]   # predicted[i] corresponds to actual[i+1]


# ── MC-Dropout Forecast with Confidence Band ───────────────────────────────────

def forecast_mc_dropout(model, last_sequence: np.ndarray,
                        scaler: MinMaxScaler,
                        n_days: int = 30,
                        n_samples: int = 100,
                        ci: float = 0.90) -> dict:
    """
    Monte-Carlo Dropout forecast — runs the model n_samples times with
    dropout ACTIVE at inference to estimate prediction uncertainty.

    Args:
        model:         Keras model (must have Dropout layers).
        last_sequence: shape (lookback, 1), SCALED values (train-scaler).
        scaler:        fitted on train only.
        n_days:        forecast horizon.
        n_samples:     MC iterations (100 is typical).
        ci:            confidence interval width (0.90 → 5th–95th percentile).

    Returns dict with keys: 'mean', 'lower', 'upper' — all shape (n_days,).
    """
    import tensorflow as tf

    all_preds = []
    for _ in range(n_samples):
        seq  = last_sequence.copy()   # (lookback, 1)
        run  = []
        for _ in range(n_days):
            x    = seq.reshape(1, len(seq), 1)
            # training=True keeps dropout active
            pred = model(x, training=True).numpy()[0, 0]
            run.append(pred)
            seq  = np.append(seq[1:], [[pred]], axis=0)
        all_preds.append(run)

    all_preds = np.array(all_preds)   # (n_samples, n_days)

    lower_q = (1 - ci) / 2
    upper_q = 1 - lower_q

    mean_scaled  = all_preds.mean(axis=0).reshape(-1, 1)
    lower_scaled = np.percentile(all_preds, lower_q * 100, axis=0).reshape(-1, 1)
    upper_scaled = np.percentile(all_preds, upper_q * 100, axis=0).reshape(-1, 1)

    return {
        "mean":  scaler.inverse_transform(mean_scaled).flatten(),
        "lower": scaler.inverse_transform(lower_scaled).flatten(),
        "upper": scaler.inverse_transform(upper_scaled).flatten(),
    }


In [ ]:
%%writefile xgboost_model.py
"""
xgboost_model.py — XGBoost baseline for stock price prediction.

Uses lag features (lag-1 … lag-N) + RSI + MACD as inputs.
No scaler leakage: fit only on train split.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

from utils import build_lag_features


def train_xgboost(close_series: pd.Series, split_idx: int,
                  n_lags: int = 60) -> tuple:
    """
    Train XGBoost on lag features from the train slice.

    Returns:
        model:   fitted XGBRegressor
        scaler:  fitted StandardScaler (for features only — target is raw price)
        cols:    feature column names (for consistent ordering at predict time)
    """
    feat_df = build_lag_features(close_series, n_lags=n_lags)

    # Align split to feature DataFrame index
    feat_train = feat_df[feat_df.index < close_series.index[split_idx]]

    X_train = feat_train.drop(columns=["close"]).values
    y_train = feat_train["close"].values

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    model = XGBRegressor(
        n_estimators   = 500,
        learning_rate  = 0.05,
        max_depth      = 6,
        subsample      = 0.8,
        colsample_bytree=0.8,
        random_state   = 42,
        n_jobs         = -1,
    )
    model.fit(X_train, y_train,
              eval_set     = [(X_train, y_train)],
              verbose      = 0)

    cols = feat_df.drop(columns=["close"]).columns.tolist()
    return model, scaler, cols


def predict_xgboost(model, close_series: pd.Series,
                    split_idx: int, scaler: StandardScaler,
                    cols: list, n_lags: int = 60) -> np.ndarray:
    """
    Predict on the test slice using the already-fitted model + scaler.
    """
    feat_df  = build_lag_features(close_series, n_lags=n_lags)
    feat_test = feat_df[feat_df.index >= close_series.index[split_idx]]

    if feat_test.empty:
        return np.array([])

    X_test = feat_test[cols].values
    X_test = scaler.transform(X_test)   # ← transform only, no re-fit
    return model.predict(X_test)


In [ ]:
%%writefile train_model.py
"""
train_model.py — Production-grade LSTM training with all ML engineering fixes.

Fixes vs v1:
  1. Scaler fit only on train split (no leakage).
  2. Walk-forward validation via TimeSeriesSplit(n_splits=5).
  3. Naive + XGBoost baseline comparison.
  4. MASE replaces MAPE.
  5. MC-Dropout 30-day forecast with confidence band.
  6. MLflow experiment tracking.
  7. Dated Parquet data snapshot for reproducibility.

Usage:
    cd D:\\stock-prediction\\backend
    python train_model.py --ticker AAPL --epochs 30 --lookback 60
"""

from __future__ import annotations
import os, argparse, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import date

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit

import mlflow
import mlflow.keras

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from utils import (
    fetch_stock_data, add_moving_averages, build_lag_features, build_technical_features, returns_to_prices,
    create_sequences, compute_metrics, naive_forecast, forecast_mc_dropout,
)
from xgboost_model import train_xgboost, predict_xgboost


# ── CLI ────────────────────────────────────────────────────────────────────────

def parse_args():
    p = argparse.ArgumentParser(description="Train LSTM stock price predictor")
    p.add_argument("--ticker",     default="AAPL")
    p.add_argument("--period",     default="5y")
    p.add_argument("--lookback",   type=int,   default=60)
    p.add_argument("--epochs",     type=int,   default=30)
    p.add_argument("--batch",      type=int,   default=32)
    p.add_argument("--n_splits",   type=int,   default=5,   help="Walk-forward CV folds")
    p.add_argument("--test_ratio", type=float, default=0.2, help="Final holdout ratio")
    p.add_argument("--mc_samples", type=int,   default=100, help="MC-Dropout iterations")
    return p.parse_args()


# ── Model Architecture ─────────────────────────────────────────────────────────

def build_lstm_model(lookback: int, num_features: int = 5) -> tf.keras.Model:
    model = Sequential([
        Input(shape=(lookback, num_features)),
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.2),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.1),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mean_squared_error")
    return model


# ── Walk-Forward Validation ────────────────────────────────────────────────────

def walk_forward_cv(df_features: np.ndarray, df_target: np.ndarray, lookback: int,
                    epochs: int, batch: int, n_splits: int) -> dict:
    """
    TimeSeriesSplit cross-validation.
    Returns per-fold metrics and the best-fold scaler + model weights path.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_metrics = []
    best_val_rmse = float("inf")
    best_fold_idx = -1

    print(f"\n  Walk-Forward CV ({n_splits} folds)")
    print("  " + "─" * 50)

    for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(df_features)):
        # Ensure enough data for sequences
        if len(train_idx) < lookback + 1 or len(val_idx) < 1:
            print(f"  Fold {fold_idx+1}: skipped (insufficient data)")
            continue

        # ── Split-safe scaler ──────────────────────────────────────────────
        train_feat = df_features[train_idx]
        train_targ = df_target[train_idx]
        val_start  = max(0, val_idx[0] - lookback)
        val_feat   = df_features[val_start: val_idx[-1] + 1]
        val_targ   = df_target[val_start: val_idx[-1] + 1]

        feat_scaler = MinMaxScaler(feature_range=(0, 1))
        targ_scaler = MinMaxScaler(feature_range=(0, 1))

        train_feat_sc = feat_scaler.fit_transform(train_feat)
        val_feat_sc   = feat_scaler.transform(val_feat)
        
        train_targ_sc = targ_scaler.fit_transform(train_targ.reshape(-1, 1))
        val_targ_sc   = targ_scaler.transform(val_targ.reshape(-1, 1))

        X_train, y_train = create_sequences(train_feat_sc, train_targ_sc, lookback)
        X_val,   y_val   = create_sequences(val_feat_sc,   val_targ_sc,   lookback)

        if len(X_train) == 0 or len(X_val) == 0:
            continue

        # ── Train ─────────────────────────────────────────────────────────
        tmp_path = f"models/_fold_{fold_idx}.keras"
        model    = build_lstm_model(lookback, num_features=df_features.shape[1])
        model.fit(
            X_train, y_train,
            epochs           = epochs,
            batch_size       = batch,
            validation_data  = (X_val, y_val),
            callbacks        = [
                EarlyStopping(monitor="val_loss", patience=4,
                              restore_best_weights=True),
                ModelCheckpoint(tmp_path, monitor="val_loss",
                                save_best_only=True, verbose=0),
                ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=3, verbose=0),
            ],
            verbose = 0,
        )

        # ── Evaluate ──────────────────────────────────────────────────────
        preds_scaled = model.predict(X_val, verbose=0)
        preds  = targ_scaler.inverse_transform(preds_scaled).flatten()
        actual = targ_scaler.inverse_transform(y_val.reshape(-1, 1)).flatten()
        naive  = naive_forecast(actual) # wait, evaluating in walk_forward_cv should also use returns_to_prices? Yes but we need last_price. To simplify, we keep it evaluating on Returns for CV folds, as RMSE of returns is also valid.
        # Align arrays: naive has 1 fewer element
        m = compute_metrics(actual[1:], preds[1:], naive)
        m["fold"] = fold_idx + 1
        fold_metrics.append(m)

        print(f"  Fold {fold_idx+1}/{n_splits} → "
              f"RMSE={m['RMSE']:.4f}  MAE={m['MAE']:.4f}  "
              f"MASE={m.get('MASE', float('nan')):.4f}  R²={m['R²']:.4f}")

        if m["RMSE"] < best_val_rmse:
            best_val_rmse = m["RMSE"]
            best_fold_idx = fold_idx

    print(f"\n  Best fold: {best_fold_idx + 1}  (RMSE={best_val_rmse:.4f})")
    return {
        "fold_metrics":  fold_metrics,
        "best_fold":     best_fold_idx,
        "best_val_rmse": best_val_rmse,
    }


# ── Main ───────────────────────────────────────────────────────────────────────

def main():
    args = parse_args()
    os.makedirs("models", exist_ok=True)
    os.makedirs("data",   exist_ok=True)
    model_path = f"models/{args.ticker}_lstm.keras"

    print(f"\n{'='*58}")
    print(f"  Stock Price Prediction  ·  {args.ticker}  ·  {date.today()}")
    print(f"{'='*58}\n")

    mlflow.set_experiment("stock-price-prediction")
    with mlflow.start_run(run_name=f"{args.ticker}_{date.today()}"):

        # ── Log hyperparameters ────────────────────────────────────────────
        mlflow.log_params({
            "ticker":     args.ticker,
            "period":     args.period,
            "lookback":   args.lookback,
            "epochs":     args.epochs,
            "batch":      args.batch,
            "n_splits":   args.n_splits,
            "test_ratio": args.test_ratio,
        })

        # 1. Fetch + cache ─────────────────────────────────────────────────
        print("[1/8] Fetching data…")
        df = fetch_stock_data(args.ticker, period=args.period, cache_dir="data")
        df = build_technical_features(df)
        
        print(f"      {len(df)} records · "
              f"{df.index[0].date()} → {df.index[-1].date()}")
        mlflow.log_param("date_range",
                         f"{df.index[0].date()}:{df.index[-1].date()}")

        features = df[["Close", "Volume", "RSI", "MACD", "Return"]].values
        target   = df["Return"].values.reshape(-1, 1)

        # 2. Holdout split (scaler leak-free) ──────────────────────────────
        print("[2/8] Splitting data (leak-free)…")
        split_idx = int(len(features) * (1 - args.test_ratio))
        train_feat = features[:split_idx]
        test_feat  = features[split_idx - args.lookback:]
        train_targ = target[:split_idx]
        test_targ  = target[split_idx - args.lookback:]

        # Fit ONLY on train
        feat_scaler = MinMaxScaler(feature_range=(0, 1))
        targ_scaler = MinMaxScaler(feature_range=(0, 1))
        
        train_feat_sc = feat_scaler.fit_transform(train_feat)
        test_feat_sc  = feat_scaler.transform(test_feat)
        
        train_targ_sc = targ_scaler.fit_transform(train_targ)
        test_targ_sc  = targ_scaler.transform(test_targ)

        X_train, y_train = create_sequences(train_feat_sc, train_targ_sc, args.lookback)
        X_test,  y_test  = create_sequences(test_feat_sc,  test_targ_sc,  args.lookback)
        print(f"      Train: {X_train.shape}  |  Test: {X_test.shape}")

        # 3. Walk-forward CV ───────────────────────────────────────────────
        print("[3/8] Walk-forward cross-validation…")
        cv_results = walk_forward_cv(
            features, target, args.lookback,
            args.epochs, args.batch, args.n_splits,
        )
        for fm in cv_results["fold_metrics"]:
            mlflow.log_metrics(
                {f"fold{fm['fold']}_rmse": fm["RMSE"],
                 f"fold{fm['fold']}_mase": fm.get("MASE", 0)},
            )
        mean_rmse = np.mean([m["RMSE"] for m in cv_results["fold_metrics"]])
        std_rmse  = np.std( [m["RMSE"] for m in cv_results["fold_metrics"]])
        print(f"\n  CV RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}")
        mlflow.log_metrics({"cv_rmse_mean": mean_rmse,
                            "cv_rmse_std":  std_rmse})

        # 4. Train final model on full train set ───────────────────────────
        print("[4/8] Training final model on full train set…")
        model = build_lstm_model(args.lookback, num_features=features.shape[1])
        history = model.fit(
            X_train, y_train,
            epochs           = args.epochs,
            batch_size       = args.batch,
            validation_split = 0.1,
            callbacks        = [
                EarlyStopping(monitor="val_loss", patience=5,
                              restore_best_weights=True),
                ModelCheckpoint(model_path, monitor="val_loss",
                                save_best_only=True),
                ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=3, verbose=1),
            ],
            verbose = 1,
        )

        # 5. Evaluate: LSTM vs Naive vs XGBoost ───────────────────────────
        print("[5/8] Evaluating models on holdout test set…")
        # LSTM (predicts Return)
        lstm_preds_scaled = model.predict(X_test, verbose=0)
        lstm_preds_return  = targ_scaler.inverse_transform(lstm_preds_scaled).flatten()
        actual_test_return = targ_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
        
        # We need absolute prices for true evaluation (dollars)
        last_train_price = df["Close"].iloc[split_idx - 1]
        
        lstm_preds  = returns_to_prices(last_train_price, lstm_preds_return)
        actual_test = returns_to_prices(last_train_price, actual_test_return)
        
        naive_preds = naive_forecast(actual_test)

        lstm_metrics = compute_metrics(
            actual_test[1:], lstm_preds[1:], naive_preds)

        # XGBoost
        print("      → Training XGBoost baseline…")
        xgb_model, xgb_scaler, xgb_cols = train_xgboost(
            df["Close"], split_idx, n_lags=args.lookback)
        xgb_preds = predict_xgboost(
            xgb_model, df["Close"], split_idx,
            xgb_scaler, xgb_cols, n_lags=args.lookback)
        n_common = min(len(actual_test), len(xgb_preds))
        
        xgb_metrics = compute_metrics(
            actual_test[1:n_common], xgb_preds[1:n_common], naive_preds[:n_common-1])

        # Naive
        naive_metrics = compute_metrics(
            actual_test[1:n_common], naive_preds[:n_common-1], naive_preds[:n_common-1])

        print("\n  ── Model Comparison ─────────────────────────────────")
        print(f"  {'Metric':8s}  {'LSTM':>10s}  {'XGBoost':>10s}  {'Naive':>10s}")
        print("  " + "─" * 45)
        for k in ["RMSE", "MAE", "R²", "MASE"]:
            l = lstm_metrics.get(k, "—")
            x = xgb_metrics.get(k, "—")
            n = naive_metrics.get(k, "—")
            fmt = lambda v: f"{v:10.4f}" if isinstance(v, float) else f"{'—':>10s}"
            print(f"  {k:8s}  {fmt(l)}  {fmt(x)}  {fmt(n)}")
        print()

        mlflow.log_metrics({
            "test_lstm_rmse": lstm_metrics["RMSE"],
            "test_lstm_mase": lstm_metrics.get("MASE", 0),
            "test_lstm_r2":   lstm_metrics["R²"],
            "test_xgb_rmse":  xgb_metrics["RMSE"],
            "test_xgb_mase":  xgb_metrics.get("MASE", 0),
            "test_naive_rmse":naive_metrics["RMSE"],
        })

        # 6. Save comparison JSON for API ─────────────────────────────────
        comparison = {
            "lstm":   lstm_metrics,
            "xgb":    xgb_metrics,
            "naive":  naive_metrics,
            "cv_rmse_mean": float(mean_rmse),
            "cv_rmse_std":  float(std_rmse),
        }
        cmp_path = f"models/{args.ticker}_comparison.json"
        with open(cmp_path, "w") as f:
            json.dump(comparison, f, indent=2)

        # 7. MC-Dropout forecast ───────────────────────────────────────────
        print("[6/8] MC-Dropout 30-day forecast…")
        last_seq   = test_scaled[-args.lookback:]   # (lookback, 1)
        mc_result  = forecast_mc_dropout(
            model, last_seq, scaler,
            n_days=30, n_samples=args.mc_samples)

        future_dates = pd.bdate_range(
            start=df.index[-1], periods=31)[1:]
        forecast_df  = pd.DataFrame({
            "date":  future_dates.astype(str),
            "mean":  mc_result["mean"].tolist(),
            "lower": mc_result["lower"].tolist(),
            "upper": mc_result["upper"].tolist(),
        })
        fc_path = f"models/{args.ticker}_forecast.json"
        forecast_df.to_json(fc_path, orient="records", indent=2)
        print(f"      30-day mean: {mc_result['mean'][:3].round(2)} …")
        print(f"      90% CI band: "
              f"[{mc_result['lower'][0]:.2f}, {mc_result['upper'][0]:.2f}] day 1")

        # 8. Save evaluation plot ─────────────────────────────────────────
        print("[7/8] Saving evaluation plots…")
        test_dates = df.index[split_idx:]
        n = min(len(test_dates), len(lstm_preds))

        fig, axes = plt.subplots(1, 2, figsize=(18, 5))
        fig.patch.set_facecolor("#0d1117")
        for ax in axes:
            ax.set_facecolor("#161b22")
            for sp in ax.spines.values(): sp.set_edgecolor("#30363d")
            ax.tick_params(colors="#c9d1d9")
            ax.xaxis.label.set_color("#c9d1d9")
            ax.yaxis.label.set_color("#c9d1d9")
            ax.title.set_color("#e6edf3")

        # Loss
        axes[0].plot(history.history["loss"],     color="#58a6ff", label="Train")
        axes[0].plot(history.history["val_loss"], color="#f78166", label="Val")
        axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch")
        axes[0].legend(facecolor="#21262d", labelcolor="#c9d1d9")

        # Actual vs models
        axes[1].plot(test_dates[:n], actual_test[:n],
                     color="#8b949e", label="Actual",   lw=1.5)
        axes[1].plot(test_dates[:n], lstm_preds[:n],
                     color="#58a6ff", label="LSTM",     lw=2)
        axes[1].plot(test_dates[:n_common], xgb_preds[:n_common],
                     color="#f0883e", label="XGBoost",  lw=1.5, ls="--")
        axes[1].set_title(f"{args.ticker} — Actual vs Models")
        axes[1].set_xlabel("Date"); axes[1].set_ylabel("Price (USD)")
        axes[1].legend(facecolor="#21262d", labelcolor="#c9d1d9")

        fig.tight_layout()
        plot_path = f"models/{args.ticker}_evaluation.png"
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(plot_path)

        # Save predictions for API
        print("[8/8] Saving predictions for API…")
        preds_df = pd.DataFrame({
            "date":      [str(d.date()) for d in df.index[split_idx: split_idx + n]],
            "actual":    actual_test[:n].tolist(),
            "lstm":      lstm_preds[:n].tolist(),
        })
        preds_df.to_json(f"models/{args.ticker}_predictions.json",
                         orient="records", indent=2)

        mlflow.keras.log_model(model, "lstm_model")

        print(f"\n✅ Done  →  {model_path}")
        print(f"   Evaluation  →  {plot_path}")
        print(f"   Forecast    →  {fc_path}")
        print(f"   Comparison  →  {cmp_path}")
        print(f"\n   Start API:  uvicorn api:app --reload")
        print(f"   Start UI:   cd ../frontend && npm run dev\n")


if __name__ == "__main__":
    main()


In [ ]:
# Install requirements
!pip install -r requirements.txt
!pip install ta # Technical analysis library

In [ ]:
# Run the model training script
!python train_model.py --ticker AAPL --epochs 30 --lookback 60